# 01 — Business Understanding, Data Audit and Splits

**Orange Belgium: Customer Churn Prediction and Retention Analytics**

**Business question:** Which customers are likely to churn, and what does historical campaign assignment suggest about retention outcomes?

**Source:** An anonymized Orange Belgium telecommunications churn/uplift dataset (`y`: churn; `t`: campaign assignment). The 160 `PC` and 18 `FACTOR` features are anonymized and do not permit customer-level causal explanations. The source CSV is deliberately not redistributed.

**Workflow:** Audit data quality, inspect treatment and churn, and establish one locked 60/20/20 stratified split. Modelling belongs in Notebook 2; treatment-effect exploration belongs in Notebook 3.

**Reproducibility:** Run from the repository root or `notebooks/` after placing the original CSV under `data/raw/` (see README).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Business Problem and Project Objectives

### Business Problem

Customer churn represents a challenge for telecommunications
companies because losing customers can reduce recurring revenue
and increase customer acquisition costs.

This project uses anonymized customer data from Orange Belgium
to investigate customer churn and retention campaign effectiveness.

### Main Objective

To develop a data-driven framework for predicting customer churn
and evaluating retention campaign interventions.

### Specific Objectives

1. Examine the structure, quality, and characteristics of
   the Orange Belgium customer dataset.

2. Investigate customer churn rates and compare outcomes
   between treatment and control groups.

3. Develop and compare machine learning models for
   predicting customer churn.

4. Evaluate whether retention interventions have different
   effects across customer segments using uplift modelling.

5. Explore how predictive and uplift models could support
   more effective customer retention decisions.

### Project Limitations

- Customer characteristics have been anonymized, limiting
  direct interpretation of individual predictors.
- The churn outcome is highly imbalanced.
- The dataset does not establish customer lifetime values
  or retention campaign costs.
- Financial recommendations will therefore require
  clearly stated assumptions.

In [2]:
from pathlib import Path
name = "Orange Belgium_churn_uplift_anonymized.csv"
roots = [Path.cwd(), Path.cwd().parent]
options = [r / folder / name for r in roots for folder in (Path("data/raw"), Path("."))]
csv_path = next((p for p in options if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError("Put the Orange Belgium source CSV in data/raw/ (see README).")
df = pd.read_csv(csv_path)
df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,FACTOR11,FACTOR12,FACTOR13,FACTOR14,FACTOR15,FACTOR16,FACTOR17,FACTOR18,y,t
0,-3.064007,-1.272259,-4.075844,1.492514,2.328039,-0.330240,-0.770389,0.035541,2.439325,-0.847266,...,V2,V2,V5,V20,V9,V11,V1,V4,0,0
1,-4.574066,-3.541815,1.107371,0.447314,-0.471140,-0.567309,0.195963,0.383654,1.523472,-0.184596,...,V2,V3,V6,V20,V9,V15,V4,V4,0,0
2,-1.751471,-2.039692,-3.788823,0.624226,0.564614,-0.713385,1.003502,1.582819,-1.461687,0.844975,...,V5,V2,V11,V20,V9,V1,V4,V4,0,1
3,-2.030089,-0.235720,-5.711960,1.071841,2.441541,0.758484,-0.534295,1.614289,-1.022909,-1.383025,...,V2,V2,V4,V20,V11,V1,V4,V4,0,1
4,-2.377857,-2.478670,-1.051946,2.812768,2.019267,0.095542,-0.033324,-1.332319,4.316526,-0.537967,...,V2,V2,V7,V20,V2,V1,V2,V2,0,1


In [3]:
# ============================================================
# PROJECT 3 - STEP 2A: EXAMINE DATASET DIMENSIONS
# ============================================================

# Extract the number of observations and variables.
n_rows, n_columns = df.shape

# Display the dataset dimensions.
print("ORANGE BELGIUM DATASET DIMENSIONS")
print("-" * 50)

print(f"Number of observations: {n_rows:,}")
print(f"Number of variables: {n_columns:,}")

# Calculate the total number of data cells.
print(f"Total data cells: {df.size:,}")

ORANGE BELGIUM DATASET DIMENSIONS
--------------------------------------------------
Number of observations: 11,896
Number of variables: 180
Total data cells: 2,141,280


In [4]:
# ============================================================
# PROJECT 3 - STEP 2B: INSPECT VARIABLE NAMES
# ============================================================

# Print all variable names in their original order.
print("DATASET VARIABLE NAMES")
print("-" * 50)

for position, column in enumerate(df.columns, start=1):
    print(f"{position}. {column}")

DATASET VARIABLE NAMES
--------------------------------------------------
1. PC1
2. PC2
3. PC3
4. PC4
5. PC5
6. PC6
7. PC7
8. PC8
9. PC9
10. PC10
11. PC11
12. PC12
13. PC13
14. PC14
15. PC15
16. PC16
17. PC17
18. PC18
19. PC19
20. PC20
21. PC21
22. PC22
23. PC23
24. PC24
25. PC25
26. PC26
27. PC27
28. PC28
29. PC29
30. PC30
31. PC31
32. PC32
33. PC33
34. PC34
35. PC35
36. PC36
37. PC37
38. PC38
39. PC39
40. PC40
41. PC41
42. PC42
43. PC43
44. PC44
45. PC45
46. PC46
47. PC47
48. PC48
49. PC49
50. PC50
51. PC51
52. PC52
53. PC53
54. PC54
55. PC55
56. PC56
57. PC57
58. PC58
59. PC59
60. PC60
61. PC61
62. PC62
63. PC63
64. PC64
65. PC65
66. PC66
67. PC67
68. PC68
69. PC69
70. PC70
71. PC71
72. PC72
73. PC73
74. PC74
75. PC75
76. PC76
77. PC77
78. PC78
79. PC79
80. PC80
81. PC81
82. PC82
83. PC83
84. PC84
85. PC85
86. PC86
87. PC87
88. PC88
89. PC89
90. PC90
91. PC91
92. PC92
93. PC93
94. PC94
95. PC95
96. PC96
97. PC97
98. PC98
99. PC99
100. PC100
101. PC101
102. PC102
103. PC103
104. PC10

In [5]:
# ============================================================
# PROJECT 3 - STEP 2C: DATA TYPES AND MISSING VALUES
# ============================================================

# Display detailed information about the dataset.
print("DATASET INFORMATION")
print("-" * 50)

df.info()

# Count variables by data type.
print("\nVARIABLE COUNTS BY DATA TYPE")
print("-" * 50)

print(df.dtypes.value_counts())

# Calculate missing values for each variable.
missing_summary = pd.DataFrame({
    "Variable": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean().values * 100
    )
})

# Sort variables by missing percentage.
missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

# Display the 20 variables with the most missing data.
print("\nTOP 20 VARIABLES WITH MISSING VALUES")
print("-" * 50)

display(missing_summary.head(20).round(2))

# Count variables containing at least one missing value.
print(
    "\nVariables with missing values:",
    (missing_summary["Missing_Count"] > 0).sum()
)

DATASET INFORMATION
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 11896 entries, 0 to 11895
Columns: 180 entries, PC1 to t
dtypes: float64(160), int64(2), str(18)
memory usage: 16.3 MB

VARIABLE COUNTS BY DATA TYPE
--------------------------------------------------
float64    160
str         18
int64        2
Name: count, dtype: int64

TOP 20 VARIABLES WITH MISSING VALUES
--------------------------------------------------


,Variable,Missing_Count,Missing_Percentage
0,PC1,0,0.0
1,PC2,0,0.0
2,PC3,0,0.0
3,PC4,0,0.0
4,PC5,0,0.0
5,PC6,0,0.0
6,PC7,0,0.0
7,PC8,0,0.0
8,PC9,0,0.0
9,PC10,0,0.0



Variables with missing values: 0


In [6]:
# ============================================================
# PROJECT 3 - STEP 2D: EXAMINE OUTCOME AND TREATMENT VARIABLES
# ============================================================

# Display the data types of the two special variables.
print("OUTCOME AND TREATMENT DATA TYPES")
print("-" * 50)

print(df[["y", "t"]].dtypes)

# ------------------------------------------------------------
# 1. EXAMINE THE OUTCOME VARIABLE
# ------------------------------------------------------------

# Count the observations in each outcome category.
print("\nOUTCOME VARIABLE (y)")
print("-" * 50)

print(df["y"].value_counts(dropna=False).sort_index())

# Calculate the percentage in each outcome category.
print("\nOutcome percentages:")

print(
    df["y"]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
    .mul(100)
    .round(2)
)

# ------------------------------------------------------------
# 2. EXAMINE THE TREATMENT VARIABLE
# ------------------------------------------------------------

# Count customers in each treatment category.
print("\nTREATMENT VARIABLE (t)")
print("-" * 50)

print(df["t"].value_counts(dropna=False).sort_index())

# Calculate treatment-group percentages.
print("\nTreatment percentages:")

print(
    df["t"]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
    .mul(100)
    .round(2)
)

# ------------------------------------------------------------
# 3. VALIDATE BINARY CODING
# ------------------------------------------------------------

# Check whether both variables contain only 0 and 1.
for column in ["y", "t"]:

    valid_values = df[column].isin([0, 1]).all()

    print(f"\n{column} contains only binary values: {valid_values}")

OUTCOME AND TREATMENT DATA TYPES
--------------------------------------------------
y    int64
t    int64
dtype: object

OUTCOME VARIABLE (y)
--------------------------------------------------
y
0    11488
1      408
Name: count, dtype: int64

Outcome percentages:
y
0    96.57
1     3.43
Name: proportion, dtype: float64

TREATMENT VARIABLE (t)
--------------------------------------------------
t
0    2886
1    9010
Name: count, dtype: int64

Treatment percentages:
t
0    24.26
1    75.74
Name: proportion, dtype: float64

y contains only binary values: True

t contains only binary values: True


In [7]:
# ============================================================
# PROJECT 3 - STEP 2E: TREATMENT AND OUTCOME RELATIONSHIP
# ============================================================

# Create a frequency table of treatment and outcome.
treatment_outcome_table = pd.crosstab(
    df["t"],
    df["y"],
    margins=True
)

print("TREATMENT AND OUTCOME FREQUENCIES")
print("-" * 50)

display(treatment_outcome_table)

# ------------------------------------------------------------
# CALCULATE THE OUTCOME RATE WITHIN EACH TREATMENT GROUP
# ------------------------------------------------------------

# Calculate the mean of the binary outcome by treatment group.
# For a variable coded 0/1, the mean equals the proportion of ones.
outcome_rate_by_treatment = (
    df.groupby("t")["y"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "Customers",
        "sum": "Outcome_1_Count",
        "mean": "Outcome_1_Rate"
    })
)

# Convert outcome rates to percentages.
outcome_rate_by_treatment["Outcome_1_Rate"] *= 100

print("\nOUTCOME RATE BY TREATMENT GROUP")
print("-" * 50)

display(outcome_rate_by_treatment.round(2))

TREATMENT AND OUTCOME FREQUENCIES
--------------------------------------------------


y,0,1,All
t,,,
0,2781,105,2886
1,8707,303,9010
All,11488,408,11896



OUTCOME RATE BY TREATMENT GROUP
--------------------------------------------------


,Customers,Outcome_1_Count,Outcome_1_Rate
t,,,
0,2886,105,3.64
1,9010,303,3.36


In [8]:
# ============================================================
# PROJECT 3 - STEP 2F: TWO-PROPORTION Z-TEST
# ============================================================

# Import the statistical testing function.
from statsmodels.stats.proportion import proportions_ztest

# ------------------------------------------------------------
# 1. EXTRACT CONTROL GROUP INFORMATION
# ------------------------------------------------------------

# Select customers assigned to the control group.
control_group = df[df["t"] == 0]

# Count the total control customers.
n_control = len(control_group)

# Count customers who churned.
churn_control = control_group["y"].sum()


# ------------------------------------------------------------
# 2. EXTRACT TREATMENT GROUP INFORMATION
# ------------------------------------------------------------

# Select customers assigned to the retention campaign.
treatment_group = df[df["t"] == 1]

# Count the total treated customers.
n_treatment = len(treatment_group)

# Count treated customers who churned.
churn_treatment = treatment_group["y"].sum()


# ------------------------------------------------------------
# 3. CALCULATE CHURN RATES
# ------------------------------------------------------------

# Churn rate among control customers.
rate_control = churn_control / n_control

# Churn rate among treated customers.
rate_treatment = churn_treatment / n_treatment

# Calculate the observed difference.
# A positive result means churn was lower in the treatment group.
rate_difference = rate_control - rate_treatment


# ------------------------------------------------------------
# 4. PERFORM THE TWO-PROPORTION Z-TEST
# ------------------------------------------------------------

# Number of churners in the two groups.
churn_counts = [
    churn_control,
    churn_treatment
]

# Total customers in each group.
sample_sizes = [
    n_control,
    n_treatment
]

# Test whether the two churn rates differ.
z_statistic, p_value = proportions_ztest(
    count=churn_counts,
    nobs=sample_sizes,
    alternative="two-sided"
)


# ------------------------------------------------------------
# 5. DISPLAY RESULTS
# ------------------------------------------------------------

print("RETENTION CAMPAIGN: TWO-PROPORTION Z-TEST")
print("-" * 55)

print(f"Control customers: {n_control:,}")
print(f"Treated customers: {n_treatment:,}")

print(f"\nControl churn rate: {rate_control * 100:.3f}%")

print(f"Treatment churn rate: {rate_treatment * 100:.3f}%")

print(
    f"\nObserved difference: "
    f"{rate_difference * 100:.3f} percentage points"
)

print(f"\nZ-statistic: {z_statistic:.4f}")
print(f"P-value: {p_value:.4f}")


# ------------------------------------------------------------
# 6. INTERPRET THE STATISTICAL TEST
# ------------------------------------------------------------

alpha = 0.05

if p_value < alpha:

    print(
        "\nDecision: Reject the null hypothesis."
    )

    print(
        "The observed churn rates differ "
        "statistically at the 5% significance level."
    )

else:

    print(
        "\nDecision: Fail to reject the null hypothesis."
    )

    print(
        "The evidence is insufficient to establish "
        "a difference between the two churn rates."
    )

RETENTION CAMPAIGN: TWO-PROPORTION Z-TEST
-------------------------------------------------------
Control customers: 2,886
Treated customers: 9,010

Control churn rate: 3.638%
Treatment churn rate: 3.363%

Observed difference: 0.275 percentage points

Z-statistic: 0.7073
P-value: 0.4794

Decision: Fail to reject the null hypothesis.
The evidence is insufficient to establish a difference between the two churn rates.


In [9]:
# ============================================================
# PROJECT 3 - STEP 2G: CONFIDENCE INTERVAL FOR CHURN DIFFERENCE
# ============================================================

# Import the function for constructing a confidence interval
# for the difference between two independent proportions.
from statsmodels.stats.proportion import confint_proportions_2indep

# ------------------------------------------------------------
# 1. CALCULATE THE 95% CONFIDENCE INTERVAL
# ------------------------------------------------------------

# Compare control churn rate minus treatment churn rate.
# We use the Newcombe method for two independent proportions.
ci_lower, ci_upper = confint_proportions_2indep(
    count1=churn_control,
    nobs1=n_control,
    count2=churn_treatment,
    nobs2=n_treatment,
    compare="diff",
    method="newcomb",
    alpha=0.05
)

# ------------------------------------------------------------
# 2. DISPLAY THE RESULTS
# ------------------------------------------------------------

print("95% CONFIDENCE INTERVAL FOR CHURN-RATE DIFFERENCE")
print("-" * 60)

# Display the observed difference in percentage points.
print(
    f"Observed difference: "
    f"{rate_difference * 100:.3f} percentage points"
)

# Display the confidence interval in percentage points.
print(
    f"95% Confidence Interval: "
    f"[{ci_lower * 100:.3f}, {ci_upper * 100:.3f}]"
    " percentage points"
)

# ------------------------------------------------------------
# 3. INTERPRET THE CONFIDENCE INTERVAL
# ------------------------------------------------------------

# A confidence interval that contains zero is compatible
# with no difference between the two population proportions.
if ci_lower <= 0 <= ci_upper:

    print("\nInterpretation:")
    print("The confidence interval includes zero.")

    print(
        "The data are compatible with both a reduction "
        "and an increase in churn under treatment."
    )

else:

    print("\nInterpretation:")
    print("The confidence interval excludes zero.")

    print(
        "The interval provides evidence of a difference "
        "between the two churn probabilities."
    )

95% CONFIDENCE INTERVAL FOR CHURN-RATE DIFFERENCE
------------------------------------------------------------
Observed difference: 0.275 percentage points
95% Confidence Interval: [-0.462, 1.102] percentage points

Interpretation:
The confidence interval includes zero.
The data are compatible with both a reduction and an increase in churn under treatment.


In [10]:
# ============================================================
# PROJECT 3 - STEP 2H: INVESTIGATE DUPLICATE OBSERVATIONS
# ============================================================

# Count rows that exactly duplicate an earlier row.
duplicate_count = df.duplicated().sum()

# Calculate the percentage of duplicate rows.
duplicate_percentage = (
    duplicate_count / len(df)
) * 100

print("DUPLICATE OBSERVATION ASSESSMENT")
print("-" * 55)

print(f"Total observations: {len(df):,}")

print(f"Duplicate observations: {duplicate_count:,}")

print(
    f"Duplicate percentage: "
    f"{duplicate_percentage:.2f}%"
)

# Check the number of unique rows.
unique_rows = len(df.drop_duplicates())

print(f"Unique observations: {unique_rows:,}")

# Do not delete duplicates at this stage.
# Identical anonymized records do not necessarily
# represent the same customer.

DUPLICATE OBSERVATION ASSESSMENT
-------------------------------------------------------
Total observations: 11,896
Duplicate observations: 0
Duplicate percentage: 0.00%
Unique observations: 11,896


In [11]:
# ============================================================
# PROJECT 3 - STEP 2I: CHECK INFINITE NUMERICAL VALUES
# ============================================================

import numpy as np
import pandas as pd

# Select numerical columns.
numerical_columns = df.select_dtypes(
    include=["number"]
).columns

# Identify positive and negative infinity.
infinite_mask = np.isinf(
    df[numerical_columns].to_numpy()
)

# Count all infinite observations.
total_infinite_values = infinite_mask.sum()

# Count infinite values separately for every numerical column.
infinite_counts = pd.Series(
    infinite_mask.sum(axis=0),
    index=numerical_columns,
    name="Infinite_Count"
)

# Keep only variables containing infinite values.
infinite_summary = infinite_counts[
    infinite_counts > 0
].sort_values(ascending=False)

print("INFINITE VALUE ASSESSMENT")
print("-" * 55)

print(
    "Number of numerical variables:",
    len(numerical_columns)
)

print(
    "Total infinite values:",
    int(total_infinite_values)
)

print(
    "Variables containing infinite values:",
    len(infinite_summary)
)

# Display affected variables when present.
if len(infinite_summary) > 0:
    display(infinite_summary.to_frame())
else:
    print("\nNo infinite numerical values detected.")

INFINITE VALUE ASSESSMENT
-------------------------------------------------------
Number of numerical variables: 162
Total infinite values: 0
Variables containing infinite values: 0

No infinite numerical values detected.


In [12]:
# ============================================================
# PROJECT 3 - STEP 2J: CATEGORICAL VARIABLE ASSESSMENT
# ============================================================

# Select categorical variables.
categorical_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns

# Count unique categories in each variable.
category_summary = pd.DataFrame({
    "Variable": categorical_columns,

    "Unique_Categories": [
        df[column].nunique(dropna=False)
        for column in categorical_columns
    ]
})

# Sort variables by the number of categories.
category_summary = category_summary.sort_values(
    by="Unique_Categories",
    ascending=False
).reset_index(drop=True)

print("CATEGORICAL VARIABLE ASSESSMENT")
print("-" * 55)

print(
    "Number of categorical variables:",
    len(categorical_columns)
)

display(category_summary)

# ------------------------------------------------------------
# IDENTIFY CONSTANT CATEGORICAL VARIABLES
# ------------------------------------------------------------

# A constant variable contains only one unique value.
constant_categories = category_summary[
    category_summary["Unique_Categories"] == 1
]

print("\nCONSTANT CATEGORICAL VARIABLES")
print("-" * 55)

if len(constant_categories) > 0:
    display(constant_categories)
else:
    print("No constant categorical variables detected.")

CATEGORICAL VARIABLE ASSESSMENT
-------------------------------------------------------
Number of categorical variables: 18


,Variable,Unique_Categories
0,FACTOR6,20
1,FACTOR4,20
2,FACTOR15,20
3,FACTOR14,20
4,FACTOR1,18
5,FACTOR16,17
6,FACTOR2,12
7,FACTOR13,12
8,FACTOR18,7
9,FACTOR17,6



CONSTANT CATEGORICAL VARIABLES
-------------------------------------------------------


,Variable,Unique_Categories
17,FACTOR3,1


In [13]:
# ============================================================
# PROJECT 3 - STEP 2K: NUMERICAL FEATURE VARIABILITY
# ============================================================

# Select only the 160 PC predictors.
# Exclude outcome (y) and treatment (t).
pc_columns = [
    column
    for column in df.columns
    if column.startswith("PC")
]

# Confirm the expected number of numerical predictors.
assert len(pc_columns) == 160

# Calculate descriptive statistics.
numerical_summary = df[pc_columns].describe().T

# Add variance.
numerical_summary["variance"] = df[pc_columns].var()

# Add skewness.
numerical_summary["skewness"] = df[pc_columns].skew()

# Display the first ten variables.
print("NUMERICAL FEATURE SUMMARY")
print("-" * 55)

display(numerical_summary.head(10).round(3))

# ------------------------------------------------------------
# IDENTIFY CONSTANT NUMERICAL FEATURES
# ------------------------------------------------------------

# Count features with exactly one observed value.
constant_numerical = [
    column
    for column in pc_columns
    if df[column].nunique() == 1
]

print("\nCONSTANT NUMERICAL FEATURES")
print("-" * 55)

print("Count:", len(constant_numerical))

print("Variables:", constant_numerical)

# ------------------------------------------------------------
# IDENTIFY LOW-VARIANCE FEATURES
# ------------------------------------------------------------

# List features with very small variance.
# This is an initial diagnostic threshold, not a
# decision to remove these predictors.
low_variance_features = (
    numerical_summary[
        numerical_summary["variance"] < 0.01
    ]
    .sort_values("variance")
)

print("\nLOW-VARIANCE FEATURES (VARIANCE < 0.01)")
print("-" * 55)

display(
    low_variance_features[
        ["mean", "std", "variance"]
    ].round(5)
)

NUMERICAL FEATURE SUMMARY
-------------------------------------------------------


,count,mean,std,min,25%,50%,75%,max,variance,skewness
PC1,11896.0,-0.0,4.268,-5.856,-2.684,-0.769,1.864,220.629,18.215,13.970
PC2,11896.0,0.0,3.593,-244.310,-1.827,-0.227,1.658,12.598,12.906,-28.616
PC3,11896.0,-0.0,3.051,-46.010,-1.868,0.278,2.042,20.669,9.306,-0.641
PC4,11896.0,0.0,2.503,-26.009,-1.268,-0.121,0.934,51.810,6.267,4.051
PC5,11896.0,0.0,2.319,-30.497,-0.967,0.141,1.296,10.458,5.380,-2.687
PC6,11896.0,0.0,2.195,-26.925,-0.984,-0.132,0.978,20.830,4.819,-0.378
PC7,11896.0,-0.0,2.105,-26.730,-0.956,-0.008,0.934,46.925,4.432,1.555
PC8,11896.0,-0.0,1.919,-23.567,-1.295,0.233,1.171,23.121,3.683,0.229
PC9,11896.0,0.0,1.906,-17.121,-1.239,-0.059,1.247,13.337,3.631,0.098
PC10,11896.0,0.0,1.842,-6.997,-1.193,-0.242,0.904,13.973,3.394,0.917



CONSTANT NUMERICAL FEATURES
-------------------------------------------------------
Count: 0
Variables: []

LOW-VARIANCE FEATURES (VARIANCE < 0.01)
-------------------------------------------------------


,mean,std,variance
PC160,0.0,0.00000,0.00000
PC159,0.0,0.00009,0.00000
PC158,-0.0,0.00165,0.00000
PC157,0.0,0.00269,0.00001
PC156,-0.0,0.00296,0.00001
PC155,-0.0,0.00310,0.00001
PC154,0.0,0.00449,0.00002
PC153,0.0,0.00657,0.00004
PC152,0.0,0.00706,0.00005
PC151,0.0,0.00823,0.00007


In [14]:
# ============================================================
# PROJECT 3 - STEP 2L: EXAMINE NUMERICAL SKEWNESS
# ============================================================

# Calculate skewness for all 160 PC predictors.
skewness_summary = pd.DataFrame({
    "Variable": pc_columns,

    "Skewness": [
        df[column].skew()
        for column in pc_columns
    ]
})

# Calculate absolute skewness for ranking.
skewness_summary["Absolute_Skewness"] = (
    skewness_summary["Skewness"].abs()
)

# Rank predictors by absolute skewness.
skewness_summary = skewness_summary.sort_values(
    by="Absolute_Skewness",
    ascending=False
)

print("TOP 15 MOST SKEWED NUMERICAL FEATURES")
print("-" * 55)

display(skewness_summary.head(15).round(3))

# Count features with substantial skewness.
highly_skewed_count = (
    skewness_summary["Absolute_Skewness"] > 1
).sum()

print(
    "\nFeatures with absolute skewness > 1:",
    highly_skewed_count
)

TOP 15 MOST SKEWED NUMERICAL FEATURES
-------------------------------------------------------


,Variable,Skewness,Absolute_Skewness
149,PC150,-53.050,53.050
147,PC148,48.844,48.844
22,PC23,46.199,46.199
158,PC159,44.216,44.216
1,PC2,-28.616,28.616
87,PC88,22.302,22.302
19,PC20,21.979,21.979
41,PC42,19.229,19.229
156,PC157,-17.938,17.938
0,PC1,13.970,13.970



Features with absolute skewness > 1: 73


In [15]:
# ============================================================
# PROJECT 3 - STEP 2M: NUMERICAL FEATURE CORRELATION ANALYSIS
# ============================================================

# ------------------------------------------------------------
# 1. CALCULATE THE CORRELATION MATRIX
# ------------------------------------------------------------

# Calculate Pearson correlations among the 160 numerical
# predictors. Do not include the outcome or treatment.
correlation_matrix = df[pc_columns].corr()

print("NUMERICAL FEATURE CORRELATION ANALYSIS")
print("-" * 55)

# Display correlations among the first ten predictors.
display(
    correlation_matrix.iloc[:10, :10].round(3)
)


# ------------------------------------------------------------
# 2. EXTRACT UNIQUE FEATURE PAIRS
# ------------------------------------------------------------

# Create a mask for the upper triangle of the matrix.
# The diagonal is excluded because a variable always
# has a correlation of 1 with itself.
upper_triangle_mask = np.triu(
    np.ones(correlation_matrix.shape, dtype=bool),
    k=1
)

# Extract correlations for unique feature pairs only.
correlation_pairs = (
    correlation_matrix
    .where(upper_triangle_mask)
    .stack()
    .reset_index()
)

# Give the resulting columns meaningful names.
correlation_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

# Calculate the absolute correlation.
correlation_pairs["Absolute_Correlation"] = (
    correlation_pairs["Correlation"].abs()
)

# Sort feature pairs from strongest to weakest.
correlation_pairs = correlation_pairs.sort_values(
    by="Absolute_Correlation",
    ascending=False
)


# ------------------------------------------------------------
# 3. DISPLAY THE STRONGEST CORRELATIONS
# ------------------------------------------------------------

print("\nTOP 15 STRONGEST FEATURE CORRELATIONS")
print("-" * 55)

display(correlation_pairs.head(15).round(4))


# ------------------------------------------------------------
# 4. COUNT STRONGLY CORRELATED FEATURE PAIRS
# ------------------------------------------------------------

strong_correlation_count = (
    correlation_pairs["Absolute_Correlation"] > 0.80
).sum()

print(
    "\nNumber of feature pairs with |r| > 0.80:",
    strong_correlation_count
)

# Display the largest absolute correlation.
print(
    "Maximum absolute correlation:",
    round(
        correlation_pairs["Absolute_Correlation"].max(),
        4
    )
)

NUMERICAL FEATURE CORRELATION ANALYSIS
-------------------------------------------------------


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
PC1,1.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0
PC2,-0.0,1.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0
PC3,-0.0,0.0,1.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0
PC4,-0.0,0.0,-0.0,1.0,0.0,0.0,-0.0,-0.0,0.0,0.0
PC5,-0.0,0.0,-0.0,0.0,1.0,-0.0,-0.0,-0.0,-0.0,-0.0
PC6,0.0,-0.0,0.0,0.0,-0.0,1.0,-0.0,-0.0,0.0,-0.0
PC7,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,1.0,-0.0,0.0,0.0
PC8,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,1.0,0.0,0.0
PC9,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,0.0,1.0,-0.0
PC10,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,1.0



TOP 15 STRONGEST FEATURE CORRELATIONS
-------------------------------------------------------


,Feature_1,Feature_2,Correlation,Absolute_Correlation
11839,PC74,PC160,-0.3340,0.3340
5919,PC37,PC160,0.2504,0.2504
14719,PC92,PC160,0.2427,0.2427
23199,PC145,PC160,-0.1874,0.1874
24159,PC151,PC160,-0.1794,0.1794
17759,PC111,PC160,-0.1586,0.1586
14559,PC91,PC160,-0.1531,0.1531
11519,PC72,PC160,0.1512,0.1512
6719,PC42,PC160,-0.1442,0.1442
11999,PC75,PC160,0.1427,0.1427



Number of feature pairs with |r| > 0.80: 0
Maximum absolute correlation: 0.334


In [16]:
# ============================================================
# PROJECT 3 - STEP 3A: PREPARE THE MODELLING DATASET
# ============================================================

# Import the required library.
import pandas as pd

# ------------------------------------------------------------
# 1. IDENTIFY THE PREDICTOR VARIABLES
# ------------------------------------------------------------

# Exclude:
# y = churn outcome
# t = treatment assignment
# FACTOR3 = constant categorical predictor
excluded_columns = ["y", "t", "FACTOR3"]

# Select all remaining predictor columns.
feature_columns = [
    column
    for column in df.columns
    if column not in excluded_columns
]

# Create the predictor matrix.
X = df[feature_columns].copy()

# ------------------------------------------------------------
# 2. SEPARATE OUTCOME AND TREATMENT
# ------------------------------------------------------------

# Customer churn outcome.
y = df["y"].copy()

# Retention campaign treatment assignment.
t = df["t"].copy()

# ------------------------------------------------------------
# 3. VERIFY THE DATASET STRUCTURE
# ------------------------------------------------------------

print("MODELLING DATASET SUMMARY")
print("-" * 55)

print(f"Total observations: {len(X):,}")

print(f"Number of predictors: {X.shape[1]}")

print(f"Outcome observations: {len(y):,}")

print(f"Treatment observations: {len(t):,}")

# Confirm that the outcome and treatment
# are not included among the predictors.
assert "y" not in X.columns
assert "t" not in X.columns
assert "FACTOR3" not in X.columns

# Confirm that all observations are aligned.
assert X.index.equals(y.index)
assert X.index.equals(t.index)

print("\nFeature preparation completed successfully.")

MODELLING DATASET SUMMARY
-------------------------------------------------------
Total observations: 11,896
Number of predictors: 177
Outcome observations: 11,896
Treatment observations: 11,896

Feature preparation completed successfully.


In [17]:
# ============================================================
# PROJECT 3 - STEP 3B: CREATE STRATIFICATION LABELS
# ============================================================

# Combine treatment and outcome values.
# Example:
# t = 0, y = 0  -> "0_0"
# t = 0, y = 1  -> "0_1"
# t = 1, y = 0  -> "1_0"
# t = 1, y = 1  -> "1_1"
stratification_labels = (
    t.astype(str)
    + "_"
    + y.astype(str)
)

# Display the distribution of the four groups.
print("STRATIFICATION GROUP DISTRIBUTION")
print("-" * 55)

print(
    stratification_labels
    .value_counts()
    .sort_index()
)

# Confirm that four groups exist.
assert stratification_labels.nunique() == 4

print("\nStratification labels created successfully.")

STRATIFICATION GROUP DISTRIBUTION
-------------------------------------------------------
0_0    2781
0_1     105
1_0    8707
1_1     303
Name: count, dtype: int64

Stratification labels created successfully.


In [18]:
# ============================================================
# PROJECT 3 - STEP 3C: TRAIN-VALIDATION-TEST SPLIT
# ============================================================

# Import the dataset splitting function.
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. CREATE THE TRAINING AND TEMPORARY DATASETS
# ------------------------------------------------------------

# First split:
# 60% training
# 40% temporary (validation + test)
#
# We split row indices so that X, y, and t remain aligned.

all_indices = df.index.to_numpy()

train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.40,
    random_state=42,
    stratify=stratification_labels
)

# ------------------------------------------------------------
# 2. SPLIT THE TEMPORARY DATA INTO VALIDATION AND TEST
# ------------------------------------------------------------

# Split the remaining 40% equally:
# 20% validation
# 20% test
#
# Use the original stratification labels corresponding
# to the temporary observations.

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42,
    stratify=stratification_labels.loc[temp_indices]
)

# ------------------------------------------------------------
# 3. CREATE THE PREDICTOR DATASETS
# ------------------------------------------------------------

# Select predictor rows for each subset.
X_train = X.loc[train_indices].copy()

X_val = X.loc[val_indices].copy()

X_test = X.loc[test_indices].copy()

# ------------------------------------------------------------
# 4. CREATE THE OUTCOME DATASETS
# ------------------------------------------------------------

# Select churn outcomes for each subset.
y_train = y.loc[train_indices].copy()

y_val = y.loc[val_indices].copy()

y_test = y.loc[test_indices].copy()

# ------------------------------------------------------------
# 5. CREATE THE TREATMENT DATASETS
# ------------------------------------------------------------

# Select retention treatment assignments.
t_train = t.loc[train_indices].copy()

t_val = t.loc[val_indices].copy()

t_test = t.loc[test_indices].copy()

# ------------------------------------------------------------
# 6. VERIFY THE DATASET SIZES
# ------------------------------------------------------------

print("TRAIN-VALIDATION-TEST SPLIT")
print("-" * 55)

print(f"Training observations: {len(X_train):,}")

print(f"Validation observations: {len(X_val):,}")

print(f"Test observations: {len(X_test):,}")

print(
    f"\nTotal observations: "
    f"{len(X_train) + len(X_val) + len(X_test):,}"
)

# Confirm that no observations were lost.
assert (
    len(X_train) + len(X_val) + len(X_test)
    == len(df)
)

# Confirm that all predictor sets have 177 columns.
assert X_train.shape[1] == 177
assert X_val.shape[1] == 177
assert X_test.shape[1] == 177

# Verify that no row appears in two different subsets.
assert set(train_indices).isdisjoint(val_indices)
assert set(train_indices).isdisjoint(test_indices)
assert set(val_indices).isdisjoint(test_indices)

print("\nDataset splitting completed successfully.")

TRAIN-VALIDATION-TEST SPLIT
-------------------------------------------------------
Training observations: 7,137
Validation observations: 2,379
Test observations: 2,380

Total observations: 11,896

Dataset splitting completed successfully.


In [19]:
# ============================================================
# PROJECT 3 - STEP 3D: VERIFY STRATIFICATION
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. CREATE A FUNCTION TO SUMMARIZE EACH DATASET
# ------------------------------------------------------------

def summarize_split(y_subset, t_subset, dataset_name):
    """
    Summarize treatment-outcome combinations
    in a particular dataset.
    """

    # Combine treatment and outcome into four groups.
    groups = (
        t_subset.astype(str)
        + "_"
        + y_subset.astype(str)
    )

    # Calculate frequency of each group.
    counts = groups.value_counts().sort_index()

    # Calculate percentage distribution.
    percentages = (
        groups.value_counts(normalize=True)
        .sort_index()
        * 100
    )

    # Combine counts and percentages.
    summary = pd.DataFrame({
        "Dataset": dataset_name,
        "Count": counts,
        "Percentage": percentages
    })

    return summary


# ------------------------------------------------------------
# 2. SUMMARIZE TRAINING, VALIDATION, AND TEST DATA
# ------------------------------------------------------------

train_summary = summarize_split(
    y_train,
    t_train,
    "Training"
)

val_summary = summarize_split(
    y_val,
    t_val,
    "Validation"
)

test_summary = summarize_split(
    y_test,
    t_test,
    "Test"
)

# Combine all summaries.
stratification_summary = pd.concat([
    train_summary,
    val_summary,
    test_summary
])

# Move the group labels into a named column.
stratification_summary.index.name = "Group"

# Display the results.
print("STRATIFICATION VERIFICATION")
print("-" * 55)

display(stratification_summary.round(2))

STRATIFICATION VERIFICATION
-------------------------------------------------------


,Dataset,Count,Percentage
Group,,,
0_0,Training,1668,23.37
0_1,Training,63,0.88
1_0,Training,5224,73.20
1_1,Training,182,2.55
0_0,Validation,556,23.37
0_1,Validation,21,0.88
1_0,Validation,1741,73.18
1_1,Validation,61,2.56
0_0,Test,557,23.40


In [24]:
# ============================================================
# PROJECT 3 - STEP 3E: FINAL DATASET SPLIT VERIFICATION
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. SUMMARIZE EACH DATASET
# ------------------------------------------------------------

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Test"],

    "Observations": [
        len(X_train),
        len(X_val),
        len(X_test)
    ],

    "Predictors": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1]
    ],

    "Churn_Events": [
        y_train.sum(),
        y_val.sum(),
        y_test.sum()
    ],

    "Churn_Rate_Percent": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ],

    "Treatment_Rate_Percent": [
        t_train.mean() * 100,
        t_val.mean() * 100,
        t_test.mean() * 100
    ]
})

# Display the summary.
print("FINAL DATASET SPLIT SUMMARY")
print("-" * 60)

display(split_summary.round(2))


# ------------------------------------------------------------
# 2. CHECK FOR MISSING VALUES
# ------------------------------------------------------------

# Verify that the predictor datasets contain no missing values.
print("\nMISSING VALUES BY DATASET")
print("-" * 60)

print("Training:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())


# ------------------------------------------------------------
# 3. VERIFY FEATURE CONSISTENCY
# ------------------------------------------------------------

# Confirm that all three datasets contain the same
# predictor columns in the same order.
assert X_train.columns.equals(X_val.columns)

assert X_train.columns.equals(X_test.columns)

print("\nFeature columns are consistent across all datasets.")


# ------------------------------------------------------------
# 4. VERIFY OUTCOME AND TREATMENT ALIGNMENT
# ------------------------------------------------------------

# Ensure each predictor dataset is correctly aligned
# with its corresponding outcome and treatment values.
assert X_train.index.equals(y_train.index)
assert X_train.index.equals(t_train.index)

assert X_val.index.equals(y_val.index)
assert X_val.index.equals(t_val.index)

assert X_test.index.equals(y_test.index)
assert X_test.index.equals(t_test.index)

print("Outcome and treatment variables are correctly aligned.")

print("\nFinal dataset verification completed successfully.")

FINAL DATASET SPLIT SUMMARY
------------------------------------------------------------


,Dataset,Observations,Predictors,Churn_Events,Churn_Rate_Percent,Treatment_Rate_Percent
0,Training,7137,177,245,3.43,75.75
1,Validation,2379,177,82,3.45,75.75
2,Test,2380,177,81,3.40,75.71



MISSING VALUES BY DATASET
------------------------------------------------------------
Training: 0
Validation: 0
Test: 0

Feature columns are consistent across all datasets.
Outcome and treatment variables are correctly aligned.

Final dataset verification completed successfully.


## Data Audit and Preparation Summary

The Orange Belgium dataset contains 11,896 observations
and 180 original variables.

### Data Quality Findings

- No missing values were detected.
- No exact duplicate observations were found.
- No infinite numerical values were identified.
- FACTOR3 was identified as a constant categorical variable.
- No numerical predictors were completely constant.
- Some numerical predictors exhibited low variance and extreme skewness.
- The numerical predictors showed limited pairwise linear correlation.

### Data Preparation Decisions

1. Retained all 11,896 observations.
2. Excluded FACTOR3 because it contains no variation.
3. Retained the 160 numerical predictors.
4. Retained the 17 nonconstant categorical predictors.
5. Separated the churn outcome (y) and treatment indicator (t).
6. Created 177 candidate predictor variables.
7. Divided the dataset into training (60%), validation (20%),
   and test (20%) subsets.
8. Stratified the split using treatment-outcome combinations.
9. Reserved the test set for final model evaluation.

### Important Considerations

The churn outcome is highly imbalanced, with only 3.43% of
customers experiencing churn.

Therefore, model evaluation will emphasize precision, recall,
F1-score, ROC-AUC, and Average Precision rather than accuracy alone.

Preprocessing transformations will be fitted using training data
only to minimize data leakage.

The dataset contains anonymized predictors, which limits direct
business interpretation of individual features.

## Notebook 1 conclusion

The audit found 11,896 customers, 180 original columns, and a rare churn outcome (408 customers; 3.43%). Excluding the constant `FACTOR3` and separating `y` and `t` leaves 177 predictors. The split is 7,137 training / 2,379 validation / 2,380 test customers and preserves the four `t × y` groups. The observed aggregate treated-minus-control churn difference was not statistically conclusive. Subsequent notebooks reuse exactly this index-based split; final test labels remain protected until predeclared evaluation.